In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

# Replace 'your_file.csv' with the actual name of your file
file_path = '/content/drive/MyDrive/capstone project/cleaned_discharge (1).csv'
file_path2 = '/content/drive/MyDrive/capstone project/mimic-iv-bhc.csv'

# Read the CSV into a DataFrame
generated = pd.read_csv(file_path)

reference = pd.read_csv(file_path2)




Mounted at /content/drive


In [2]:
generated.head()

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-DS-21,10000032,22595853,DS,21,2180-05-07 00:00:00,2180-05-09 15:26:00,Name: ** Unit No: ** Admission Date: ** Discha...
1,10000032-DS-22,10000032,22841357,DS,22,2180-06-27 00:00:00,2180-07-01 10:15:00,Name: ** Unit No: ** Admission Date: ** Discha...
2,10000032-DS-23,10000032,29079034,DS,23,2180-07-25 00:00:00,2180-07-25 21:42:00,Name: ** Unit No: ** Admission Date: ** Discha...
3,10000032-DS-24,10000032,25742920,DS,24,2180-08-07 00:00:00,2180-08-10 05:43:00,Name: ** Unit No: ** Admission Date: ** Discha...
4,10000084-DS-17,10000084,23052089,DS,17,2160-11-25 00:00:00,2160-11-25 15:09:00,Name: ** Unit No: ** Admission Date: ** Discha...


In [3]:
reference.head()

,note_id,input,target,input_tokens,target_tokens
0,10000032-DS-21,<SEX> F <SERVICE> MEDICINE <ALLERGIES> No Know...,"___ HCV cirrhosis c/b ascites, hiv on ART, h/o...",1946,231
1,10000032-DS-22,<SEX> F <SERVICE> MEDICINE <ALLERGIES> Percoce...,"___ with HIV on HAART, HCV cirrhosis with asci...",2183,810
2,10000117-DS-21,<SEX> F <SERVICE> MEDICINE <ALLERGIES> omepraz...,Ms. ___ is a ___ with history of GERD who pres...,1060,172
3,10000117-DS-22,<SEX> F <SERVICE> ORTHOPAEDICS <ALLERGIES> ome...,The patient presented to the emergency departm...,1195,330
4,10000248-DS-10,<SEX> M <SERVICE> MEDICINE <ALLERGIES> No Know...,Mr. ___ is a ___ with history of mild FVIII de...,1961,230


In [4]:
common_ids = pd.Series(list(set(generated['note_id']).intersection(set(reference['note_id']))))

# Step 3: Randomly sample 1000 common noted_ids
sample_ids = common_ids.sample(1000, random_state=42)

# Step 4: Filter both datasets for those noted_ids
subset_generated = generated[generated['note_id'].isin(sample_ids)].sort_values("note_id")
subset_reference = reference[reference['note_id'].isin(sample_ids)].sort_values("note_id")

In [5]:
!pip install transformers torch google-cloud-storage pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 135.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 114.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvji

In [6]:
!pip install transformers torch sentencepiece

In [7]:
from transformers import PegasusTokenizer, PegasusForConditionalGeneration
import torch
from tqdm import tqdm  # For progress bar

# Load model and tokenizer
model_name = "google/pegasus-cnn_dailymail"
tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)



# Store summaries
summaries = []

# Loop through each text entry
for text in tqdm(subset_generated['text'], desc="Generating Summaries"):
    # Tokenize input
    inputs = tokenizer(text, truncation=True, padding="longest", return_tensors="pt", max_length=1024).to(device)

    # Generate summary
    with torch.no_grad():
        summary_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            num_beams=5,
            max_length=150,
            early_stopping=True
        )

    # Decode summary
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    summaries.append(summary)

# Add summaries to your DataFrame
subset_generated["summary"] = summaries


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]


Generating Summaries: 100%|██████████| 1000/1000 [31:08<00:00,  1.87s/it]


In [8]:
pd.set_option('display.max_colwidth', None)
subset_generated["summary"].head(5)


,summary
593,Year-old gravida 0 experienced postmenopausal bleeding that led to a pelvic ultrasound at ** Ultrasound .<n>She has a history of bilateral borderline ovarian cancer and is status post a left salpingo-oophorectomy and right ovarian cystectomy in ** .
663,"Mr. ** is a ** year old male with a history of Coronary Artery Disease Postcath in ** showing 3VD that has been managed medically .<n>Angina is described as variable, times starting in neck and radiating down to b/l chest, pressure sensation, rarely down L arm .<n>On ** morning while lying in bed he developed cough productive of pink sputum and was short of breath ."
710,"Mr ** is a ** withHistory OfAlzheimer's dementia, atrial fibrillation on Coumadin, colon cancer with metastatic disease to the liver, who p/w concern for bowel obstruction ** cancer.<n>He was diagnosed with colon cancer relatively recently. He had family have opted for minimally invasive approach and he has not had chemotherapy, surgery, or radiation."
1931,"Patient was eating dinner at home when he suddenly became unresponsive .<n>His daughter came into the room and checked his pulse, which was noted to be in the ** .<n>EP was consulted and interrogated the patient's pacemaker, which was functioning normally ."
2947,"His symptoms occured after urinating 7am, were gradual onset, no radiation, no sob, palpitation, diaphoresis, but pt does not increased heart rate and bilateral hand numbness .<n>Patient was admitted to medical service for evaluation of chest pain in setting of recent indeterminant Endotracheal Tube ."


In [9]:
subset_generated.to_csv("new_sub_generated_summary.csv", index=False)


In [10]:
!pip install -q evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.


In [11]:
!pip install rouge_score nltk

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=45bca5583a3aab92b1eb4ccf57479d294526a61b30e4648c713f13cd4df46390
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


In [12]:
import evaluate  # Make sure this is imported


rouge = evaluate.load("rouge")

def compute_metrics(predictions, references):
    rouge_scores = rouge.compute(predictions=predictions, references=references)
    return {'ROUGE': rouge_scores}

# Calculate metrics
metrics = compute_metrics(subset_generated['summary'].tolist(), subset_reference['target'].tolist())
print("Evaluation Metrics:", metrics)

Evaluation Metrics: {'ROUGE': {'rouge1': np.float64(0.1574770080568203), 'rouge2': np.float64(0.06768557922157753), 'rougeL': np.float64(0.11571509854935363), 'rougeLsum': np.float64(0.11583001835930411)}}


In [13]:
import evaluate

# Load BLEU metric
bleu = evaluate.load("bleu")

# Extract summaries (as raw strings)
predictions = subset_generated["summary"].tolist()
references = subset_reference["target"].tolist()

# BLEU expects: predictions = list of strings
#               references = list of list of strings
# So wrap each reference in a list
formatted_references = [[ref] for ref in references]

# Compute BLEU
bleu_results = bleu.compute(predictions=predictions,
                            references=formatted_references)

# Print result
print("🔹 BLEU Score")
print(f"BLEU: {bleu_results['bleu']:.4f}")


🔹 BLEU Score
BLEU: 0.0003


In [14]:
!pip install -q evaluate bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00


In [15]:


import evaluate

# Load BERTScore
bertscore = evaluate.load("bertscore")

# Extract summaries
predictions = subset_generated["summary"].tolist()
references = subset_reference["target"].tolist()

# Compute BERTScore
bert_results = bertscore.compute(predictions=predictions,
                                 references=references,
                                 lang="en")

# Print results
print("🔹 BERTScore Results")
print(f"Precision: {sum(bert_results['precision']) / len(bert_results['precision']):.4f}")
print(f"Recall:    {sum(bert_results['recall']) / len(bert_results['recall']):.4f}")
print(f"F1 Score:  {sum(bert_results['f1']) / len(bert_results['f1']):.4f}")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔹 BERTScore Results
Precision: 0.8392
Recall:    0.7924
F1 Score:  0.8149
